# CXR-Recall: retrieval encoder comparison

**Question:** is the image encoder the bottleneck for retrieval? Retrieval with the current encoder finds a neighbour sharing a finding only ~20% of the time on abnormal queries (1.5x chance), even though every tagged query has hundreds of possible matches in the gallery (oracle P@5 = 1.0). This notebook runs the **same retrieval evaluation** with several encoders, on the same split, gallery and queries, and changes nothing else.

| Encoder | Pretraining | Fine-tuned on these labels? |
|---|---|---|
| `imagenet_vit_cls` / `_patchmean` | ViT-B/16, ImageNet-1K | No |
| `arm1_vit_cls` (**current system**) / `_patchmean` | ViT-B/16, ImageNet-1K | Yes (Arm 1, seed 0) |
| `xrv_densenet_chex` | DenseNet-121, CheXpert only (TorchXRayVision) | No |
| `raddino_cls` / `_patchmean` | ViT-B/14 DINOv2, chest X-rays (Microsoft RAD-DINO) | No |

`_cls` uses the [CLS] token, `_patchmean` the mean of the final patch tokens.

**Evaluation** (Section 6): gallery = the 5,977 train images, queries = val (746) and test (743), cosine top-5. Relevance = shares at least one of the 14 tags, as everywhere else in the project. Reported for all queries and stratified (normal / abnormal with a tag / no tag), with the random baseline and the oracle ceiling, plus MRR and Jaccard@5. Each encoder is compared with the current system on test using a paired bootstrap and Wilcoxon test. Nothing is tuned, so val and test are both just reported.

**Test-set contamination, checked on purpose:**
- TorchXRayVision: use **`densenet121-res224-chex`** (CheXpert only). Do **not** switch to `densenet121-res224-all`, which was trained on several datasets **including Open-I**, this notebook's test data.
- RAD-DINO: its model card lists MIMIC-CXR, CheXpert, NIH ChestX-ray14, PadChest and BRAX, not Open-I. Re-check the card if the model is updated.

**Settings:** GPU on, **Internet on** (pip install and model downloads). Attach "Chest X-rays Indiana University" (`raddar/chest-xrays-indiana-university`). Use **Save Version, Save & Run All**. Runtime is about 1-1.5 h on a T4: Arm 1 ~47 min, extraction ~15-20 min. If one encoder fails to download, it's recorded as an error and the others still run.

In [ ]:
!pip install -q torchxrayvision

In [ ]:
import os, glob, json, time, copy, random, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 1. Configuration

In [ ]:
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'
ARM1_SEED = 0                 # same seed as the multi-seed run's seed 0
EPOCHS_REAL, PATIENCE = 20, 5
BATCH_SIZE, LR = 16, 1e-4
EXTRACT_BATCH = 32
PRETRAINED = True
K = 5
N_BOOTSTRAP = 10000
REFERENCE = 'arm1_vit_cls'    # the encoder the current system uses
XRV_WEIGHTS = 'densenet121-res224-chex'   # CheXpert only -- NOT '-all' (includes Open-I)
RADDINO_ID = 'microsoft/rad-dino'
ENCODERS = ['imagenet_vit', 'arm1_vit', 'xrv_densenet_chex', 'raddino']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Locate the attached dataset (identical to the multi-seed notebook)

In [ ]:
import os

candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached — use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

## 3. Manifest, label vocabulary, fixed study-level split (identical to the multi-seed notebook)

In [ ]:
reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_report_text(row):
    parts = [str(row.get(FINDINGS_COL, '')), str(row.get(IMPRESSION_COL, ''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan').strip()

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'report_text': merged.apply(build_report_text, axis=1),
    'mesh_tags': merged.apply(build_tags, axis=1),
})

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
manifest.head()

In [ ]:
from collections import Counter
import numpy as np

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

In [ ]:
def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

EXPECTED_VOCAB = ['normal', 'Lung', 'Opacity', 'Cardiomegaly', 'Pulmonary Atelectasis', 'Calcinosis',
                  'Calcified Granuloma', 'Thoracic Vertebrae', 'Cicatrix', 'Spine', 'Pleural Effusion',
                  'Aorta', 'Markings', 'Diaphragm']
if INPUT_DIR == '/kaggle/input':
    assert (len(train_df), len(val_df), len(test_df)) == (5977, 746, 743), 'split differs from every other Phase 1 run'
    assert label_vocab == EXPECTED_VOCAB, 'label vocabulary differs from every other Phase 1 run'
    print('split and vocabulary match the documented Phase 1 runs')

## 4. Arm 1 (current system's encoder), trained once with seed 0

Verbatim Arm 1 recipe from the multi-seed notebook. GPU kernels aren't bit-for-bit deterministic, so this won't exactly reproduce multi-seed seed 0 (test 0.7099); its own test AUROC is printed for reference.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class IUXrayDataset(Dataset):
    def __init__(self, df, image_root, transform, label_vocab):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform
        self.label_vocab = label_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
        image = self.transform(image)
        tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
        label = mesh_tags_to_vector(tags, self.label_vocab)
        return image, label

def tags_to_vec(tag_str):
    return mesh_tags_to_vector(tag_str.split(';') if tag_str else [], label_vocab)

In [ ]:
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

class NoRetrievalClassifier(nn.Module):
    def __init__(self, num_labels, pretrained=True):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.backbone = vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        self.head = nn.Linear(768, num_labels)

    def forward(self, images):
        return self.head(self.backbone(images))

transform = ViT_B_16_Weights.DEFAULT.transforms()
train_ds = IUXrayDataset(train_df, IMAGE_ROOT, transform, label_vocab)
val_ds = IUXrayDataset(val_df, IMAGE_ROOT, transform, label_vocab)
test_ds = IUXrayDataset(test_df, IMAGE_ROOT, transform, label_vocab)  # held out: never used for early stopping or model selection
print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, LR = 16, 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

criterion = nn.BCEWithLogitsLoss()

def macro_auroc(y_true, y_score):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, i], y_score[:, i]))
    return float(np.mean(scores)) if scores else float('nan')

def held_out_auroc(forward_fn, loader):
    """One pass over a held-out loader with the model already in eval() mode.
    forward_fn(batch_tuple) -> logits. Every loader in this notebook yields
    labels at position 1. Used for the TEST numbers: val is what every arm's
    early stopping selects on, so val AUROC is an optimistically-selected
    number; test is evaluated exactly once per arm, on the reloaded best
    checkpoint, and never influences any training or selection decision."""
    all_labels, all_scores = [], []
    with torch.no_grad():
        for batch in loader:
            all_scores.append(torch.sigmoid(forward_fn(batch)).cpu().numpy())
            all_labels.append(batch[1].numpy())
    return macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))

print('setup complete -- device:', device, ' batch size:', BATCH_SIZE, ' lr:', LR)

In [ ]:
def run_arm1(seed):
    set_seed(seed)
    model = NoRetrievalClassifier(num_labels=len(label_vocab), pretrained=PRETRAINED).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_auroc, best_state, best_epoch, bad = -1.0, None, 0, 0
    for epoch in range(1, EPOCHS_REAL + 1):
        model.train()
        total = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total += loss.item() * images.size(0)
        model.eval()
        val_auroc = held_out_auroc(lambda b: model(b[0].to(device)), val_loader)
        print(f'[Arm 1 seed {seed}] epoch {epoch}/{EPOCHS_REAL}  train_loss={total / len(train_ds):.4f}  val_macro_auroc={val_auroc:.4f}')
        if val_auroc > best_auroc:
            best_auroc, best_state, best_epoch, bad = val_auroc, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'[Arm 1] stopping early at epoch {epoch}')
                break
    model.load_state_dict(best_state)
    model.eval()
    test_auroc = held_out_auroc(lambda b: model(b[0].to(device)), test_loader)
    print(f'[Arm 1] best val = {best_auroc:.4f} (epoch {best_epoch})   TEST = {test_auroc:.4f}')
    return model, {'arm1_val': round(best_auroc, 4), 'arm1_test': round(test_auroc, 4), 'arm1_epoch': best_epoch}

t0 = time.time()
arm1_model, arm1_info = run_arm1(ARM1_SEED)
torch.save(arm1_model.state_dict(), os.path.join(WORK_DIR, 'arm1_seed0.pt'))
print(f'Arm 1 done in {(time.time() - t0) / 60:.1f} min; checkpoint saved to arm1_seed0.pt')

## 5. Feature extractors

Every extractor returns `{variant_name: (N, D) array}` plus the label matrix, for one split, in the split's row order.

In [ ]:
class PathLabelDataset(Dataset):
    """Image + 14-label vector, with an arbitrary per-encoder preprocessing function."""
    def __init__(self, df, preprocess):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        x = self.preprocess(os.path.join(IMAGE_ROOT, row['image_path']))
        return x, tags_to_vec(row['mesh_tags'])

def loader_for(df, preprocess):
    return DataLoader(PathLabelDataset(df, preprocess), batch_size=EXTRACT_BATCH, shuffle=False, num_workers=2)

vit_tf = ViT_B_16_Weights.DEFAULT.transforms()
def vit_preprocess(path):
    return vit_tf(Image.open(path).convert('RGB'))

def vit_tokens(vit, x):
    """Final-layer tokens of a torchvision ViT: (B, 1 + n_patches, 768). Mirrors VisionTransformer.forward."""
    x = vit._process_input(x)
    cls = vit.class_token.expand(x.shape[0], -1, -1)
    return vit.encoder(torch.cat([cls, x], dim=1))

@torch.no_grad()
def extract_vit(vit, df, prefix):
    vit.eval()
    out = {f'{prefix}_cls': [], f'{prefix}_patchmean': []}
    labels, checked = [], False
    for x, y in loader_for(df, vit_preprocess):
        x = x.to(device)
        tok = vit_tokens(vit, x)
        if not checked:  # [CLS] must equal the backbone's own forward output
            assert torch.allclose(tok[:, 0], vit(x), atol=1e-4), 'token extraction does not match vit.forward'
            checked = True
        out[f'{prefix}_cls'].append(tok[:, 0].cpu().numpy())
        out[f'{prefix}_patchmean'].append(tok[:, 1:].mean(1).cpu().numpy())
        labels.append(y.numpy())
    return {k: np.concatenate(v) for k, v in out.items()}, np.concatenate(labels)

In [ ]:
def make_xrv():
    import torchxrayvision as xrv
    import torchvision
    model = xrv.models.DenseNet(weights=XRV_WEIGHTS).to(device).eval()
    tf = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(224)])
    def preprocess(path):
        img = np.array(Image.open(path).convert('L')).astype(np.float32)
        img = xrv.datasets.normalize(img, 255)[None, ...]   # (1, H, W), range [-1024, 1024]
        return torch.from_numpy(tf(img).astype(np.float32))
    return model, preprocess

@torch.no_grad()
def extract_xrv(model, preprocess, df):
    feats, labels = [], []
    for x, y in loader_for(df, preprocess):
        f = F.relu(model.features(x.to(device)))
        feats.append(F.adaptive_avg_pool2d(f, 1).flatten(1).cpu().numpy())
        labels.append(y.numpy())
    return {'xrv_densenet_chex': np.concatenate(feats)}, np.concatenate(labels)

In [ ]:
def make_raddino():
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained(RADDINO_ID)
    model = AutoModel.from_pretrained(RADDINO_ID).to(device).eval()
    def preprocess(path):
        return processor(images=Image.open(path).convert('RGB'), return_tensors='pt')['pixel_values'][0]
    return model, preprocess

@torch.no_grad()
def extract_raddino(model, preprocess, df):
    n_reg = getattr(model.config, 'num_register_tokens', 0) or 0
    out = {'raddino_cls': [], 'raddino_patchmean': []}
    labels = []
    for x, y in loader_for(df, preprocess):
        o = model(pixel_values=x.to(device))
        out['raddino_cls'].append(o.pooler_output.cpu().numpy())
        out['raddino_patchmean'].append(o.last_hidden_state[:, 1 + n_reg:].mean(1).cpu().numpy())
        labels.append(y.numpy())
    return {k: np.concatenate(v) for k, v in out.items()}, np.concatenate(labels)

## 6. Retrieval evaluation

Gallery = train split. For each query: cosine top-5, then P@5 (fraction of the 5 sharing at least one tag), MRR and Jaccard@5. The random baseline is the fraction of the gallery sharing a tag with the query, and the oracle is min(matches, 5) / 5. Strata: `normal` (tagged normal), `abnormal` (at least one tag, not normal), `untagged` (none of the 14 tags, can never match).

In [ ]:
NORMAL_IDX = label_vocab.index('normal') if 'normal' in label_vocab else None

def strata(q_lbl):
    untagged = q_lbl.sum(1) == 0
    normal = (q_lbl[:, NORMAL_IDX] == 1) if NORMAL_IDX is not None else np.zeros(len(q_lbl), bool)
    return {'all': np.ones(len(q_lbl), bool), 'tagged': ~untagged, 'normal': normal,
            'abnormal': ~untagged & ~normal, 'untagged': untagged}

def per_query_retrieval(g_emb, g_lbl, q_emb, q_lbl, k=K):
    g = g_emb / np.linalg.norm(g_emb, axis=1, keepdims=True)
    q = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    sims = q @ g.T
    nb = np.argsort(-sims, axis=1, kind='stable')[:, :k]
    nl = g_lbl[nb]                                     # (n, k, L)
    inter = (nl * q_lbl[:, None, :]).sum(-1)
    union = ((nl + q_lbl[:, None, :]) > 0).sum(-1)
    shared = inter > 0
    first = np.where(shared.any(1), shared.argmax(1), -1)
    return {'p_at_k': shared.mean(1),
            'mrr': np.where(first >= 0, 1.0 / (first + 1), 0.0),
            'jaccard': (inter / np.maximum(union, 1)).mean(1),
            'top1_sim': sims[np.arange(len(q)), nb[:, 0]]}

def reference_rows(g_lbl, q_lbl, k=K):
    matches = (q_lbl @ g_lbl.T > 0).sum(1)
    return {'random': matches / len(g_lbl), 'oracle': np.minimum(matches, k) / k}

def summarise(pq, q_lbl):
    s = strata(q_lbl)
    row = {f'P@5_{name}': float(pq['p_at_k'][m].mean()) for name, m in s.items() if m.any()}
    row['MRR_all'] = float(pq['mrr'].mean())
    row['Jaccard_abnormal'] = float(pq['jaccard'][s['abnormal']].mean()) if s['abnormal'].any() else float('nan')
    row['median_top1_cos'] = float(np.median(pq['top1_sim']))
    return row

def paired_bootstrap(diffs, n=N_BOOTSTRAP, seed=42):
    rng = np.random.RandomState(seed)
    means = np.array([diffs[rng.randint(0, len(diffs), len(diffs))].mean() for _ in range(n)])
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(lo), float(hi), float(min(1.0, 2 * min((means <= 0).mean(), (means >= 0).mean())))

## 7. Extract and evaluate every encoder

In [ ]:
embeddings, errors, info = {}, {}, {}
split_dfs = {'train': train_df, 'val': val_df, 'test': test_df}

def run_encoder(name):
    t = time.time()
    if name == 'imagenet_vit':
        vit = vit_b_16(weights=ViT_B_16_Weights.DEFAULT if PRETRAINED else None).to(device)
        vit.heads = nn.Identity()
        fn = lambda df: extract_vit(vit, df, 'imagenet_vit')
    elif name == 'arm1_vit':
        fn = lambda df: extract_vit(arm1_model.backbone, df, 'arm1_vit')
    elif name == 'xrv_densenet_chex':
        m, pre = make_xrv()
        fn = lambda df: extract_xrv(m, pre, df)
    elif name == 'raddino':
        m, pre = make_raddino()
        fn = lambda df: extract_raddino(m, pre, df)
    else:
        raise ValueError(name)
    per_split = {s: fn(df) for s, df in split_dfs.items()}
    for variant in per_split['train'][0]:
        embeddings[variant] = {s: per_split[s][0][variant] for s in split_dfs}
        np.savez_compressed(os.path.join(WORK_DIR, f'emb_{variant}.npz'),
                            **{f'{s}_emb': per_split[s][0][variant] for s in split_dfs},
                            **{f'{s}_lbl': per_split[s][1] for s in split_dfs})
    labels_by_split = {s: per_split[s][1] for s in split_dfs}
    print(f'{name}: {list(per_split["train"][0])} dims '
          f'{[v.shape[1] for v in per_split["train"][0].values()]} in {(time.time() - t) / 60:.1f} min')
    return labels_by_split

LABELS = None
for name in ENCODERS:
    try:
        lbl = run_encoder(name)
        if LABELS is None:
            LABELS = lbl
        else:  # every encoder must see the same rows in the same order
            assert all(np.array_equal(LABELS[s], lbl[s]) for s in split_dfs), f'{name}: label order mismatch'
    except Exception as e:
        errors[name] = f'{type(e).__name__}: {e}'
        print(f'!! {name} FAILED -- {errors[name]}')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('variants extracted:', list(embeddings), ' errors:', errors)

In [ ]:
results, per_query = {}, {}
for split in ('val', 'test'):
    g_lbl, q_lbl = LABELS['train'], LABELS[split]
    ref = reference_rows(g_lbl, q_lbl)
    s = strata(q_lbl)
    rows = {'random baseline': {f'P@5_{n}': float(ref['random'][m].mean()) for n, m in s.items() if m.any()},
            'oracle (ceiling)': {f'P@5_{n}': float(ref['oracle'][m].mean()) for n, m in s.items() if m.any()}}
    for variant, emb in embeddings.items():
        pq = per_query_retrieval(emb['train'], g_lbl, emb[split], q_lbl)
        per_query[(variant, split)] = pq
        rows[variant] = summarise(pq, q_lbl)
    results[split] = pd.DataFrame(rows).T
    print(f'\n===== {split.upper()} ({len(q_lbl)} queries; strata sizes: '
          f'{ {n: int(m.sum()) for n, m in s.items()} }) =====')
    print(results[split].round(4).to_string())

In [ ]:
print(f'TEST, paired against the current system ({REFERENCE}); diff = encoder - reference\n')
comparisons = {}
if REFERENCE in embeddings:
    s = strata(LABELS['test'])
    for variant in embeddings:
        if variant == REFERENCE:
            continue
        comparisons[variant] = {}
        for stratum in ('tagged', 'abnormal'):
            m = s[stratum]
            d = per_query[(variant, 'test')]['p_at_k'][m] - per_query[(REFERENCE, 'test')]['p_at_k'][m]
            lo, hi, p_boot = paired_bootstrap(d)
            try:
                from scipy.stats import wilcoxon
                p_w = float(wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
            except Exception:
                p_w = float('nan')
            sig = 'SIGNIFICANT' if lo > 0 or hi < 0 else 'not significant'
            comparisons[variant][stratum] = {'diff': round(float(d.mean()), 4), 'ci': [round(lo, 4), round(hi, 4)],
                                             'p_wilcoxon': p_w, 'p_boot': p_boot}
            print(f'{variant:22s} {stratum:9s} n={m.sum():3d}  diff={d.mean():+.4f}  '
                  f'95% CI [{lo:+.4f}, {hi:+.4f}]  Wilcoxon p={p_w:.4g}  -> {sig}')
else:
    print('reference encoder missing -- see errors above')

In [ ]:
summary = {'arm1': arm1_info, 'errors': errors,
           'test': results['test'].round(4).to_dict(orient='index'),
           'val': results['val'].round(4).to_dict(orient='index'),
           'test_vs_reference': comparisons}
with open(os.path.join(WORK_DIR, 'encoder_comparison_results.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print('RESULT_JSON:', json.dumps(summary, default=float))

## How to read this

- **Main column: `P@5_abnormal`.** It covers the clinically interesting queries, and its ceiling is 1.0. `P@5_all` is dragged down by untagged queries (ceiling ~0.81) and propped up by easy normal-to-normal matches.
- **Is the encoder the bottleneck?** Compare each medical encoder against `arm1_vit_cls` in the paired test. A significant gain on `abnormal`, even though the medical encoders were never fine-tuned on these labels, means the ImageNet backbone is the limit. If nothing beats it, the limit is the labels and data, not the backbone.
- **[CLS] vs patch mean** isolates the pooling choice within one encoder.
- **`median_top1_cos`** shows how compressed each space is. Values near 1.0 mean everything looks alike to that encoder.
- Paste the `RESULT_JSON` line (or the two tables and the comparison block) back into the chat.